# Libraries

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import flwr as fl
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, log_loss

FEATURE_COLS = [
    "frame.len",
    "ip.proto",
    "ip.checksum",
    "ip.tos",
    "tcp.srcport",
    "tcp.dstport",
    "tcp.flags",
    "tcp.window_size_value",
    "tcp.window_size_scalefactor",
    "tcp.checksum",
    "tcp.pdu.size",
    "udp.srcport",
    "udp.dstport",
]

# client training and sending updates

In [ ]:


def load_folder_csv(folder_path):
    all_dfs = []
    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".csv"):
            path = os.path.join(folder_path, filename)
            try:
                df = pd.read_csv(path)
                all_dfs.append(df)
                print(f"Loaded {filename} shape={df.shape}")
            except Exception as e:
                print(f"Failed to load {filename}: {e}")
    if not all_dfs:
        raise ValueError("No CSV files found in the provided folder.")
    return pd.concat(all_dfs, ignore_index=True)


def preprocess_features(df):
    X = df.copy()

    def conv(x):
        if pd.isna(x):
            return 0.0
        if isinstance(x, str) and x.startswith("0x"):
            try:
                return int(x, 16)
            except:
                return 0.0
        try:
            return float(x)
        except:
            return 0.0

    for c in X.columns:
        X[c] = X[c].apply(conv)
    return X


class NetworkTrafficClient(fl.client.NumPyClient):
    def __init__(self, folder_path):
        print("Initializing client...")
        self.df = load_folder_csv(folder_path)
        missing = [c for c in FEATURE_COLS if c not in self.df.columns]
        if missing:
            raise ValueError(f"Missing required feature columns in CSVs: {missing}")
        self.X = preprocess_features(self.df[FEATURE_COLS])
        if "label" not in self.df.columns:
            raise ValueError("CSV files must include a 'label' column.")
        self.raw_labels = self.df["label"].values
        self.model = None
        self.le = None
        self.initialized = False

    def _read_classes_from_config(self, config):
        # Expect classes to be provided as JSON string under key "classes_json".
        if "classes_json" in config:
            try:
                cls_list = json.loads(config["classes_json"])
                return np.array(cls_list)
            except Exception as e:
                print("Warning: failed to parse classes_json:", e)
                return None
        return None

    def get_parameters(self, config):
        return [
            self.model.coef_.astype(np.float32),
            self.model.intercept_.astype(np.float32),
        ]

    def set_parameters(self, parameters):
        coef, intercept = parameters
        self.model.coef_ = np.array(coef, dtype=np.float64)
        self.model.intercept_ = np.array(intercept, dtype=np.float64)

    def fit(self, parameters, config):
        print(">>> fit() called by server. config keys:", list(config.keys()))
        try:
            if not self.initialized:
                classes_arr = self._read_classes_from_config(config)
                if classes_arr is not None:
                    self.le = LabelEncoder()
                    self.le.classes_ = classes_arr
                elif self.le is None:
                    # If the client has no knowledge of classes, raise a helpful error
                    raise ValueError(
                        "No classes provided via config and no local label encoder available."
                    )
                # transform labels (will raise if a label is unseen)
                self.y = self.le.transform(self.raw_labels)
                self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
                    self.X, self.y, test_size=0.2, random_state=42
                )
                n_features = int(config.get("n_features", self.X.shape[1]))
                self.model = SGDClassifier(
                    loss="log_loss",
                    max_iter=1,
                    tol=None,
                    warm_start=True,
                    random_state=42,
                )
                classes = np.arange(len(self.le.classes_))
                self.model.partial_fit(
                    np.zeros((1, n_features)), np.array([0]), classes=classes
                )
                self.initialized = True
                print("Client initialized inside fit(): data and model ready.")

            # set parameters and perform one partial_fit epoch
            self.set_parameters(parameters)
            X_train_arr = np.asarray(self.X_train, dtype=np.float64)
            y_train_arr = np.asarray(self.y_train, dtype=np.int64)
            print(f"Training locally on {len(X_train_arr)} examples...")
            self.model.partial_fit(X_train_arr, y_train_arr)
            print("Local training done; returning parameters to server.")
            return (
                [
                    self.model.coef_.astype(np.float32),
                    self.model.intercept_.astype(np.float32),
                ],
                len(X_train_arr),
                {},
            )
        except Exception as e:
            import traceback

            traceback.print_exc()
            print("Exception in fit():", repr(e))
            raise

    def evaluate(self, parameters, config):
        print(">>> evaluate() called by server. config keys:", list(config.keys()))
        try:
            # Initialize model if needed, but prefer classes from config
            classes_arr = self._read_classes_from_config(config)
            if self.model is None:
                if classes_arr is not None:
                    self.le = LabelEncoder()
                    self.le.classes_ = classes_arr
                elif self.le is None:
                    raise ValueError(
                        "No classes provided via config and no local label encoder available for evaluate."
                    )
                self.model = SGDClassifier(
                    loss="log_loss",
                    max_iter=1,
                    tol=None,
                    warm_start=True,
                    random_state=42,
                )
                self.model.partial_fit(
                    np.zeros((1, int(config.get("n_features", self.X.shape[1])))),
                    np.array([0]),
                    classes=np.arange(len(self.le.classes_)),
                )

            # set server-provided parameters
            self.set_parameters(parameters)

            X_test_arr = np.asarray(self.X_test, dtype=np.float64)
            y_test_arr = np.asarray(self.y_test, dtype=np.int64)
            print(f"Evaluating on {len(X_test_arr)} examples...")
            y_proba = self.model.predict_proba(X_test_arr)
            loss = float(
                log_loss(y_test_arr, y_proba, labels=np.arange(len(self.le.classes_)))
            )
            y_pred = self.model.predict(X_test_arr)
            acc = float(accuracy_score(y_test_arr, y_pred))
            print(f"Evaluation done. loss={loss:.4f}, acc={acc:.4f}")
            return float(loss), len(X_test_arr), {"accuracy": acc}
        except Exception as e:
            import traceback

            traceback.print_exc()
            print("Exception in evaluate():", repr(e))
            raise


if __name__ == "__main__":
    folder_path = input("Path to folder with CSV(s): ").strip()
    server_ip = (
        input("Server IP (for same machine use 127.0.0.1): ").strip() or "127.0.0.1"
    )
    server_port = input("Server port (default 8080): ").strip() or "8080"
    server_address = f"{server_ip}:{server_port}"

    client = NetworkTrafficClient(folder_path)
    fl_client = client.to_client()
    print(f"Connecting to Flower server at {server_address} ...")
    fl.client.start_client(server_address=server_address, client=fl_client)
